<a href="https://colab.research.google.com/github/sheikhahmed468-crypto/Contact-Management-System/blob/main/orax_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import os

print("Please upload dataset")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

if 'fake reviews dataset (1).csv' in os.listdir('.') and 'fake reviews dataset.csv' not in os.listdir('.'):
    os.rename('fake reviews dataset (1).csv', 'fake reviews dataset.csv')
    print("Renamed 'fake reviews dataset (1).csv' to 'fake reviews dataset.csv' for consistency.")

Please upload 'fake reviews dataset.csv' below:


Saving fake reviews dataset.csv to fake reviews dataset.csv
User uploaded file "fake reviews dataset.csv" with length 15284091 bytes


In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving fake reviews dataset.csv to fake reviews dataset (1).csv
User uploaded file "fake reviews dataset (1).csv" with length 15284091 bytes


                                                  **Problem Definition**
In today's e-commerce landscape, online reviews play a crucial role in influencing consumer purchasing decisions. However, the proliferation of fake reviews whether computer-generated CG or human-written but deceptive opinion spam, OR undermines consumer trust and distort market competition. The ability to accurately identify and filter out such fraudulent reviews is paramount for maintaining the integrity of online platforms and ensuring a fair marketplace. This project aims to address the challenge of distinguishing between genuine and fake product reviews.

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer # New import for lemmatization
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
try:
    nltk.data.find('corpora/omw-1.4')
except LookupError:
    nltk.download('omw-1.4')

#       here we load the dataset
# Assuming the uploaded file is 'fake reviews dataset.csv'
df = None # Initialize df to None
try:
    df = pd.read_csv('fake reviews dataset.csv')
    print("Dataset loaded successfully.")
    print("Original DataFrame head:")
    print(df.head())
    print("\nDataFrame info:")
    df.info()
except FileNotFoundError:
    print("Error: 'fake reviews dataset.csv' not found. Please ensure the file is uploaded.")

# Determine text and label columns. Assuming 'text_' and 'label' based on common datasets.
text_column = 'text_'
label_column = 'label'

# Check if df was loaded and has the necessary columns before proceeding
if df is None:
    print(" Please check the file upload and run the cell again.")
elif text_column not in df.columns or label_column not in df.columns:
    print(f"\nError: Expected columns '{text_column}' and '{label_column}' not found.")
    print(f"Available columns: {df.columns.tolist()}")
    print("Please modify 'text_column' and 'label_column' variables to match your dataset.")
else:
    #      Data Cleaning and Preprocessing
    #      Handle missing values
    print(f"\nMissing values before cleaning:\n{df.isnull().sum()}")
    df.dropna(subset=[text_column, label_column], inplace=True)
    print(f"Missing values after cleaning:\n{df.isnull().sum()}")

    # Text Cleaning Function with Lemmatization
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    def clean_text(text):
        text = str(text).lower()  # here we Convert to string and lowercase
        text = re.sub(r'[^a-z\s]', '', text)  # Removing non-alphabetic characters
        # Removing stopwords and lemmatize
        text = ' '.join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])
        return text

    df['cleaned_text'] = df[text_column].apply(clean_text)
    print("\nText cleaning complete (with lemmatization). Sample original vs cleaned text:")
    print(df[[text_column, 'cleaned_text']].head())

    #  Split Data
    X = df['cleaned_text']
    y = df[label_column]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"\nData split into training ({(len(X_train)/len(df))*100:.2f}%) and testing ({(len(X_test)/len(df))*100:.2f}%).")

    # Feature Extraction (TF-IDF) with N-grams
    tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2)) # Added ngram_range for unigrams and bigrams

    X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
    X_test_tfidf = tfidf_vectorizer.transform(X_test)
    print(f"\nTF-IDF features extracted (with unigrams and bigrams). Number of features: {X_train_tfidf.shape[1]}")

    # Train Logistic Regression Model
    logistic_model = LogisticRegression(max_iter=1000, random_state=42)
    logistic_model.fit(X_train_tfidf, y_train)
    print("\nLogistic Regression model trained successfully.")

    #  Evaluate Model Performance
    y_pred = logistic_model.predict(X_test_tfidf)

    print("\n Model Evaluation (Logistic Regression)")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"F1-Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Dataset loaded successfully.
Original DataFrame head:
             category  rating label  \
0  Home_and_Kitchen_5       5    CG   
1  Home_and_Kitchen_5       5    CG   
2  Home_and_Kitchen_5       5    CG   
3  Home_and_Kitchen_5       1    CG   
4  Home_and_Kitchen_5       5    CG   

                                               text_  
0  Love this!  Well made, sturdy, and very comfor...  
1  love it, a great upgrade from the original.  I...  
2  This pillow saved my back. I love the look and...  
3  Missing information on how to use it, but it i...  
4  Very nice set. Good quality. We have had the s...  

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40432 entries, 0 to 40431
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   category  40432 non-null  object
 1   rating    40432 non-null  int64 
 2   label     40432 non-null  object
 3   text_     40432 non-null  object
dtypes: int64(1), object(3)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.metrics import Precision, Recall
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os


file_path = 'fake reviews dataset.csv'

try:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} not found. Please ensure the upload step finished.")

    df = pd.read_csv(file_path)
    print(f"✅ Dataset loaded from: {file_path}")

    print("\n--- Deep Learning Model (LSTM) ---")

    #  Prepare Data
    X_raw = df['text_']
    y_labels_raw = df['label']

    X_train_raw, X_test_raw, y_train_labels_dl, y_test_labels_dl = train_test_split(
        X_raw, y_labels_raw, test_size=0.2, random_state=42, stratify=y_labels_raw
    )

    label_encoder_dl = LabelEncoder()
    y_train_encoded = label_encoder_dl.fit_transform(y_train_labels_dl)
    y_test_encoded = label_encoder_dl.transform(y_test_labels_dl)

    #  Tokenization
    vocab_size = 10000
    tokenizer = Tokenizer(num_words=vocab_size, oov_token='<unk>')
    tokenizer.fit_on_texts(X_train_raw)

    X_train_padded = pad_sequences(tokenizer.texts_to_sequences(X_train_raw), maxlen=500, padding='post', truncating='post')
    X_test_padded = pad_sequences(tokenizer.texts_to_sequences(X_test_raw), maxlen=500, padding='post', truncating='post')

    #  Model Architecture
    lstm_model = Sequential([
        Embedding(vocab_size, 128),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(64)),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', Precision(), Recall()])

    # Train
    print("Training... this may take a few minutes.")
    lstm_model.fit(
        X_train_padded, y_train_encoded,
        epochs=5, batch_size=64, validation_split=0.1,
        callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
        verbose=1
    )

    # Evaluate
    y_pred_probs_lstm = lstm_model.predict(X_test_padded)
    y_pred_lstm = (y_pred_probs_lstm > 0.5).astype(int)

    lstm_accuracy = accuracy_score(y_test_encoded, y_pred_lstm)
    print(f"\nLSTM Accuracy: {lstm_accuracy:.4f}")
    print(classification_report(y_test_encoded, y_pred_lstm, target_names=label_encoder_dl.classes_))

except Exception as e:
    print(f" Error: {e}")

✅ Dataset loaded from: fake reviews dataset.csv

--- Deep Learning Model (LSTM) ---
Training... this may take a few minutes.
Epoch 1/5
455/455 ━━━━━━━━━━━━━━━━━━━━ 612s 1s/step - accuracy: 0.9089 - loss: 0.2228 - precision: 0.9098 - recall: 0.9077 - val_accuracy: 0.9400 - val_loss: 0.1518 - val_precision: 0.9128 - val_recall: 0.9728
Epoch 2/5
455/455 ━━━━━━━━━━━━━━━━━━━━ 612s 1s/step - accuracy: 0.9549 - loss: 0.1247 - precision: 0.9518 - recall: 0.9584 - val_accuracy: 0.9453 - val_loss: 0.1360 - val_precision: 0.9234 - val_recall: 0.9709
Epoch 3/5
455/455 ━━━━━━━━━━━━━━━━━━━━ 610s 1s/step - accuracy: 0.9770 - loss: 0.0679 - precision: 0.9771 - recall: 0.9769 - val_accuracy: 0.9536 - val_loss: 0.1434 - val_precision: 0.9633 - val_recall: 0.9430
Epoch 4/5
455/455 ━━━━━━━━━━━━━━━━━━━━ 622s 1s/step - accuracy: 0.9858 - loss: 0.0433 - precision: 0.9856 - recall: 0.9861 - val_accuracy: 0.9515 - val_loss: 0.1497 - val_precision: 0.9472 - val_recall: 0.9560
253/253 ━━━━━━━━━━━━━━━━━━━━ 41s 15

In [ ]:
import google.generativeai as genai
import os
import pandas as pd
from google.colab import userdata
from sklearn.preprocessing import LabelEncoder
import numpy as np
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("✅ Google API key loaded and configured.")
except Exception as e:
    print(f"Configuration Error: {e}")
    GOOGLE_API_KEY = None

# Check if previous DL dependencies exist
required_vars = ['label_encoder_dl', 'y_test_encoded', 'y_pred_lstm', 'X_test_raw']
missing_vars = [v for v in required_vars if v not in globals()]

# Only proceed if API key is configured
if not GOOGLE_API_KEY:
    print("LLM integration skipped: No API key configured.")
else: # API key is present
    # List available models to debug 'gemini-1.0-pro' not found issue
    print("\n     Listing available models    ")
    found_gemini_pro = False
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"Found model: {m.name}")
            if "gemini-pro" in m.name or "gemini-2.5-pro" in m.name:
                found_gemini_pro = True
                # Use the exact name found in the list
                actual_gemini_pro_model_name = m.name
                print(f"Identified primary Gemini Pro model as: {actual_gemini_pro_model_name}")
                break

    if not found_gemini_pro:
        print(" Did not find a suitable 'gemini-pro' model for generateContent. LLM explanation will be skipped.")
        # If no Gemini Pro is found, we should prevent the explanation part from running
        GOOGLE_API_KEY = None # Effectively skip the rest of the block

    if GOOGLE_API_KEY and missing_vars:
        print(f" Missing DL variables: {missing_vars}")
        print("Attempting to define dummy variables for demonstration purposes if original LSTM variables are not found.")
        try:
            # Define dummy variables to allow the code to run without crashing,
            # but the LLM explanations will not be based on actual model predictions.
            # This assumes two classes 'CG', 'OR' as per the sampling logic
            label_encoder_dl = LabelEncoder()
            label_encoder_dl.fit(['CG', 'OR'])

            dummy_num_samples = 2
            X_test_raw = pd.Series([
                "This is a dummy review for a positively coded product.",
                "This is another dummy review for a negatively coded product."
            ])
            y_test_encoded = np.array([0, 1]) # Assuming 0 for CG, 1 for OR
            y_pred_lstm = np.array([[0], [1]]) # Assuming predictions match true labels for these dummies

            print("✅ Dummy DL variables defined to allow LLM explanation code to run.")
            # If dummy variables are successfully created, clear missing_vars to proceed
            missing_vars = []
        except Exception as dummy_e:
            print(f" Failed to define dummy variables: {dummy_e}")
            print("LLM integration skipped: Could not set up dummy variables. Please ensure the LSTM training cell (5PLCayUM1Dgp) has finished and its variables are in scope.")
            # Keep missing_vars as it is to prevent the LLM explanation block from running

    # After potentially creating dummy variables and checking for model availability, re-check missing_vars and API Key
    if GOOGLE_API_KEY and not missing_vars: # This means either real variables are present, or dummies were successfully created, and Gemini Pro is available
        try:
            # Initializing model with the correct model name identified above
            model_genai = genai.GenerativeModel(actual_gemini_pro_model_name)
            print(f"{actual_gemini_pro_model_name} model loaded.")

            true_labels_map = label_encoder_dl.inverse_transform(y_test_encoded)
            predicted_labels_map = label_encoder_dl.inverse_transform(y_pred_lstm.flatten())

            test_results_df = pd.DataFrame({
                'text': X_test_raw.reset_index(drop=True),
                'true_label': true_labels_map,
                'predicted_label': predicted_labels_map
            })

            print("\n     Generating LLM Explanations ")

            # Sample a few cases (One correct CG and one correct OR)
            cg_indices = test_results_df[(test_results_df['true_label'] == 'CG') & (test_results_df['predicted_label'] == 'CG')].index
            or_indices = test_results_df[(test_results_df['true_label'] == 'OR') & (test_results_df['predicted_label'] == 'OR')].index

            if len(cg_indices) > 0 and len(or_indices) > 0:
                sample_indices = [cg_indices[0], or_indices[0]]

                for idx in sample_indices:
                    review_text = test_results_df.loc[idx, 'text']
                    true_label = test_results_df.loc[idx, 'true_label']
                    predicted_label = test_results_df.loc[idx, 'predicted_label']

                    prompt = f"""Review: '{review_text}'\nPredicted Label: '{predicted_label}'\nTrue Label: '{true_label}'\n\nExplain briefly why a machine learning model might classify this as {predicted_label}. Focus on linguistic cues or patterns common in {predicted_label} reviews."""

                    print(f"\n      Review Sample ---\n{review_text[:200]}")
                    response = model_genai.generate_content(prompt)
                    print(f"Explanation: {response.text}")
            else:
                print("Could not find matching samples for both CG and OR for explanation (even with dummy data).")

        except Exception as e:
            print(f"Error during generation: {e}")

✅ Google API key loaded and configured.

--- Listing available models ---
Found model: models/gemini-2.5-flash
Found model: models/gemini-2.5-pro
Identified primary Gemini Pro model as: models/gemini-2.5-pro
models/gemini-2.5-pro model loaded.

--- Generating LLM Explanations ---

--- Review Sample ---
"MINIMAL" Attractactant, IF ANY... (Cats are very picky...


Error during generation: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro
Please retry in 15.131870842s.


In [ ]:
from google.colab import userdata
try:
    key = userdata.get('GOOGLE_API_KEY')
    print("✅ Success: GOOGLE_API_KEY found in Secrets!")
except Exception as e:
    print(" Error:", e)
    print("Check the 🔑 icon on the left and ensure 'Notebook access' is turned on for GOOGLE_API_KEY.")

✅ Success: GOOGLE_API_KEY found in Secrets!
